In [ ]:
import pandas as pd
import numpy as np
import anndata
from collections import defaultdict
import gc
import warnings
warnings.filterwarnings('ignore')
import os

In [ ]:
hvg_genes = pd.read_csv('/jdata/qmy/VirtualCell/tahoe_100M/hvg_all_genes.txt', header=None)[0].tolist()
hvg_set = set(hvg_genes)
print(f"HVG genes: {len(hvg_genes)}")

In [ ]:
concs = [ '0.05','0.5', '5.0']
conc_map = { 0.05: '0.05', 0.5: '0.5', 5.0: '5.0'}

paths = [f"/jdata/qmy/VirtualCell/tahoe_100M/plate{i}_filt_Vevo_Tahoe100M_WServicesFrom_ParseGigalab.h5ad" for i in range(1, 15)]

for plate_idx, p in enumerate(paths):
    print(f"Processing plate {plate_idx+1}/14: {p}")
    drug_cell_means = {c: defaultdict(lambda: np.zeros(len(hvg_genes), dtype=np.float32)) for c in concs}
    drug_cell_counts = {c: defaultdict(int) for c in concs}
    
    ad = anndata.read_h5ad(p, backed='r')
    var_names = ad.var_names.tolist()
    hvg_indices = [i for i, g in enumerate(var_names) if g in hvg_set]
    cell_names = ad.obs['cell_name'].values
    drug_strings = ad.obs['drugname_drugconc'].values
    parsed_drugs = []
    for i, drug_str in enumerate(drug_strings):
        if i % 500000 == 0 and i > 0:
            print(f"  Parsed {i}/{len(drug_strings)} cells")
        if isinstance(drug_str, str) and drug_str:
            try:
                drugs = eval(drug_str)
                # 将浓度转换为字符串
                cell_drugs = [(d, conc_map[c]) for d, c, _ in drugs if c in conc_map]
                parsed_drugs.append(cell_drugs)
            except:
                parsed_drugs.append([])
        else:
            parsed_drugs.append([])
    batch_size = 2000
    total_batches = (len(cell_names) + batch_size - 1) // batch_size
    for batch_idx, start in enumerate(range(0, len(cell_names), batch_size)):
        end = min(start + batch_size, len(cell_names))
        
        if batch_idx % 10 == 0:
            print(f"  Batch {batch_idx+1}/{total_batches}, cells {start}-{end}")
        
        expr_all = ad[start:end, :].X
        if hasattr(expr_all, 'toarray'):
            expr_all = expr_all.toarray().astype(np.float32)
        
        expr = expr_all[:, hvg_indices]
        
        for j in range(end - start):
            cell_name = cell_names[start + j]
            cell_drugs = parsed_drugs[start + j]
            
            if not cell_drugs:
                continue
            
            expr_row = expr[j]
            
            for drug, conc_str in cell_drugs:
                key = (drug, cell_name)
                drug_cell_means[conc_str][key] += expr_row
                drug_cell_counts[conc_str][key] += 1
        
        del expr_all, expr
        if batch_idx % 20 == 0:
            gc.collect()
    
    ad.file.close()
    for conc in concs:
        data = []
        index = []
        
        for (drug, cell), sum_expr in drug_cell_means[conc].items():
            count = drug_cell_counts[conc][(drug, cell)]
            if count > 0:
                data.append(sum_expr / count)
                index.append((drug, cell))
        
        if data:
            df = pd.DataFrame(np.array(data), index=pd.MultiIndex.from_tuples(index, names=['drug', 'cell']))
            df.columns = hvg_genes
            df.to_csv(f'plate{plate_idx+1}_drug_cell_means_{conc}uM.csv')
    del drug_cell_means, drug_cell_counts, parsed_drugs
    gc.collect()